# Notebook 02: Web Scraping & Text Extraction

**Time:** 30 minutes  
**Prerequisites:** Notebook 01 complete  
**Goal:** Extract clean text from web pages using traditional and modern tools

This notebook will:
1. Extract text from web pages with trafilatura (traditional approach)
2. Use Crawl4AI for LLM-ready markdown extraction (modern 2025 approach)
3. Scrape arXiv paper abstracts for a pretraining dataset
4. Compare extraction quality between tools

> **Why this matters:** The quality of LLM pretraining data starts with extraction. Garbage in, garbage out. Modern tools like Crawl4AI (50K+ GitHub stars) produce cleaner, LLM-optimized markdown directly, while trafilatura remains a solid baseline for simpler extraction tasks.

In [27]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

# Week 3 specific imports
import src.scraping_utils
importlib.reload(src.scraping_utils)
from src.scraping_utils import (
    extract_with_trafilatura,
    scrape_arxiv_abstracts,
    compare_extractors,
)

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 02")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Setup complete -- ready for Notebook 02


---

## Part 1: Common Crawl & trafilatura

Common Crawl is a massive public archive of web pages -- like a snapshot of the entire internet, updated monthly. It's used to train models like GPT, Claude, and LLaMA.

**trafilatura** is a Python library that extracts the main content from web pages, stripping away navigation, ads, and boilerplate. Think of it as using a highlighter on millions of web pages to extract just the important sentences.

In [23]:
print("=" * 65)
print("Experiment 1: trafilatura Text Extraction")
print("=" * 65)
print()

# Extract text from an arXiv abstract page
url = "https://arxiv.org/abs/2404.00001"
result = extract_with_trafilatura(url)

print(f"\nExtracted text preview:")
print(result['text'][:500] if result['text'] else 'No text extracted')

Experiment 1: trafilatura Text Extraction

[trafilatura] Fetching: https://arxiv.org/abs/2404.00001
  Extracted 2,366 chars in 0.6s

Extracted text preview:
Physics > Physics Education
[Submitted on 3 Feb 2024]
Title:Uso de herramientas digitales matemáticas en la Educación Secundaria
View PDF HTML (experimental)Abstract:Information and Community Technologies (ICT) are very present in our society nowadays and particularly in the educative field. In just two decades, we have passed from a learning based, in many cases, on the master lessons to one such that methodologies like the flipped classroom or the gamification are stronger than ever. Along thi


In [24]:
print("=" * 65)
print("Experiment 2: Extract from a Blog Post")
print("=" * 65)
print()

# Try a blog/news article
blog_url = "https://ai.meta.com/blog/meta-llama-4/"
blog_result = extract_with_trafilatura(blog_url)

print(f"\nExtracted {blog_result['char_count']:,} chars")
print(f"Preview: {blog_result['text'][:400]}..." if blog_result['text'] else 'No text extracted')

Experiment 2: Extract from a Blog Post

[trafilatura] Fetching: https://ai.meta.com/blog/meta-llama-4/
  Extracted 120 chars in 2.4s

Extracted 120 chars
Preview: You might find what you need on our homepage .
Foundational models
Our approach
Research
Meta AI
Latest news
Meta © 2026...


In [25]:
# TODO 1: Extract text from a URL of your choice
#
# Pick a web page related to your interests (blog post, docs page, wiki article).
# Extract the text and evaluate the quality.

# my_url = "[STUDENT: PASTE YOUR URL HERE]"
my_url = "https://en.wikipedia.org/wiki/Large_language_model"

print("=" * 65)
print("TODO 1: Custom URL Extraction")
print("=" * 65)
print()

my_result = extract_with_trafilatura(my_url)
print(f"\nExtracted {my_result['char_count']:,} chars")
print(f"Preview: {my_result['text'][:300]}..." if my_result['text'] else 'No text extracted')

todo1_reflection = """
[YOUR REFLECTION HERE]

- What URL did you choose and why?
- I choose wikipedia page about large language models, because it is a topic of interest to me and I wanted to see how well trafilatura can extract information from a structured page with multiple sections.
- How well did trafilatura extract the main content?
- Because it's extracted from wiki and I think the quality is pretty good. And the main reason I think is that wiki is always extracted as training data for LLM so trafilatura is likely optimized for it.
- Were there any missing or incorrect parts in the extraction?
- I think the extraction is pretty good, but there are some missing parts such as the references and the infobox on the right side of the page. However, the main content of the article is well preserved.
"""

print()
print(todo1_reflection)

TODO 1: Custom URL Extraction

[trafilatura] Fetching: https://en.wikipedia.org/wiki/Large_language_model
  Extracted 93,719 chars in 2.2s

Extracted 93,719 chars
Preview: Large language model
A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can generate, summarize, translate and parse text in many contexts, and are a foundational technology behind modern chatb...


[YOUR REFLECTION HERE]

- What URL did you choose and why?
- I choose wikipedia page about large language models, because it is a topic of interest to me and I wanted to see how well trafilatura can extract information from a structured page with multiple sections.
- How well did trafilatura extract the main content?
- Because it's extracted from wiki and I think the quality is pretty good. And the main reason I think is that wiki is always extracted as training data for LLM so trafilatura is likely optimized for it.

---

## Part 2: Markdown Extraction -- Crawl4AI & html2text (2025)

The key insight of modern web scraping for LLMs: **output Markdown, not plain text**. Structured Markdown preserves headings, links, lists, and code blocks -- all of which help LLMs understand document structure.

**Crawl4AI** (50K+ GitHub stars) is the leading tool for this. It uses a headless browser to render JavaScript-heavy pages and outputs clean, LLM-ready Markdown. However, it requires **Python 3.10+**.

For Python 3.9 environments, we use **html2text** as a fallback -- it converts HTML to Markdown without a browser, which works great for static pages like arXiv.

| Tool | Output | JS Support | Python | Best For |
|------|--------|-----------|--------|----------|
| **trafilatura** | Plain text | No | 3.7+ | Bulk extraction, removing boilerplate |
| **html2text** | Markdown | No | 3.6+ | Static pages, lightweight Markdown |
| **Crawl4AI** | Markdown | Yes (browser) | 3.10+ | JS-heavy sites, LLM pipelines |

```
pip install html2text        # Lightweight fallback (works everywhere)
pip install crawl4ai         # Full-featured (requires Python 3.10+)
```

In [29]:
print("=" * 65)
print("Experiment 3: Markdown Extraction vs trafilatura")
print("=" * 65)
print()

# Compare both extractors on the same page
# Uses Crawl4AI if Python 3.10+, otherwise html2text as markdown fallback
comparison_url = "https://arxiv.org/abs/2404.00001"

try:
    comparison = compare_extractors(comparison_url)
    
    print("\n--- trafilatura output (plain text, first 300 chars) ---")
    print(comparison['trafilatura']['text'][:300])
    
    md_result = comparison['crawl4ai']
    print(f"\n--- {md_result.get('method', 'markdown')} output (first 300 chars) ---")
    print(md_result['text'][:300])
    
    print(f"\nKey difference: trafilatura gives plain text ({comparison['trafilatura']['char_count']:,} chars)")
    print(f"Markdown extractor preserves structure ({md_result['char_count']:,} chars)")
except Exception as e:
    print(f"Comparison failed: {e}")
    print("TIP: pip install html2text  (or pip install crawl4ai for Python 3.10+)")

Experiment 3: Markdown Extraction vs trafilatura

EXTRACTOR COMPARISON
URL: https://arxiv.org/abs/2404.00001

[trafilatura] Fetching: https://arxiv.org/abs/2404.00001


Task was destroyed but it is pending!
task: <Task pending name='Task-91' coro=<Connection.run() running at c:\Users\wesle\Desktop\HW\wesley_HW3\Homework3-Submission\.venv\Lib\site-packages\playwright\_impl\_connection.py:314> wait_for=<Future pending cb=[Task.__wakeup()]>>


  Extracted 2,366 chars in 0.7s
[Crawl4AI] Fetching: https://arxiv.org/abs/2404.00001


[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 1.30s 

[SCRAPE].. ◆ https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 0.09s 

[COMPLETE] ● https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 1.42s 

  Extracted 8,575 chars in 3.6s

--- Comparison ---
  trafilatura:  2,366 chars in 0.7s
  crawl4ai: 8,575 chars in 3.6s

--- trafilatura output (plain text, first 300 chars) ---
Physics > Physics Education
[Submitted on 3 Feb 2024]
Title:Uso de herramientas digitales matemáticas en la Educación Secundaria
View PDF HTML (experimental)Abstract:Information and Community Technologies (ICT) are very present in our society nowadays and particularly in the educative field. In just

--- crawl4ai output (first 300 chars) ---
[Skip to main content](https://arxiv.org/abs/2404.00001#content)
[![Cornell University](https://arxiv.org/static/browse/0.3.4/images/icons/cu/cornell-reduced-white-SMALL.svg)](https://www.cornell.edu/)
[Learn about arXiv becoming an independent nonprofit.](https://tech.cornell.edu/arxiv/)
We gratefu

Key difference: trafilatura gives plain text (2,366 chars)
Markdown extractor preserves structure (8,575 chars)


In [31]:
# TODO 2: Analyze the differences between extractors
#
# Use the LLM to analyze the quality differences between
# trafilatura and Crawl4AI outputs.

print("=" * 65)
print("TODO 2: LLM Analysis of Extraction Quality")
print("=" * 65)
print()

traf_text = blog_result['text'][:500] if blog_result['text'] else 'No extraction'

start = time.time()
response = client.generate(
    prompt=f"""Compare these two web extraction approaches for building LLM pretraining datasets:

1. **trafilatura** (traditional, 2020): Extracts plain text from HTML, removes boilerplate.
   Sample output: {traf_text[:200]}

2. **Crawl4AI** (modern, 2025): Produces LLM-ready Markdown with structure preserved.

For a team building a pretraining dataset:
- Which tool would you recommend for different scenarios?
- What are the trade-offs (speed, quality, features)?
- When would you use one over the other?""",
    system="You are an expert in data engineering for LLM pretraining.",
    max_tokens=400,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

todo2_reflection = """
[YOUR REFLECTION HERE]

- Which extractor produced better output for your test URLs?
- I think Crawl4AI produced better output for the arXiv page because it preserved the structure and formatting of the original page, which is important for understanding the content. However, for the blog post, trafilatura did a decent job of extracting the main text, but it lost some of the formatting and structure that could be useful for downstream tasks.
- For what types of pages would you prefer Crawl4AI over trafilatura?
- I would prefer Crawl4AI for pages that have a lot of structure, such as research papers, documentation, or any page where the formatting (headings, lists, code blocks) is important for understanding the content. For simpler pages or when I just need the main text without formatting, trafilatura might be sufficient.
- How does output format (plain text vs Markdown) affect downstream LLM usage?
- The output format can significantly affect downstream LLM usage. Plain text is simpler and may be sufficient for tasks that only require the main content, but it loses important structural information that can help the LLM understand the context and relationships between different parts of the text. Markdown, on the other hand, preserves this structure, which can improve the performance of the LLM on tasks like summarization, question answering, or any task that benefits from understanding the hierarchy and formatting of the content.
"""

print()
print(todo2_reflection)

TODO 2: LLM Analysis of Extraction Quality

Response in 9.0s
Model: claude-sonnet-4-6
Tokens: 182 in, 400 out
Stop reason: max_tokens
# Web Extraction for LLM Pretraining: trafilatura vs Crawl4AI

## Critical Upfront Observation

The sample output you showed **actually demonstrates trafilatura failing**, not succeeding:

```
You might find what you need on our homepage.
Foundational models
Our approach
Research
Meta AI
Latest news
Meta © 2026
```

This is **navigation boilerplate** — exactly what trafilatura is supposed to remove. This is important context for the comparison.

---

## Honest Tool Assessment

### trafilatura (2020)
```python
import trafilatura

html = fetch_url("https://example.com/article")
text = trafilatura.extract(html)
# Returns: plain text, boilerplate removed (when working correctly)
```

**What it actually does well:**
- Fast, lightweight, battle-tested on news/article content
- Excellent for high-volume pipeline processing
- Good boilerplate removal on well-str

---

## Part 3: Building an arXiv Paper Dataset

Let's build a small pretraining dataset by scraping scientific paper abstracts from arXiv. This mirrors what real pretraining pipelines do at scale -- Meta's LLaMA 4 used academic papers as a key data source.

In [32]:
print("=" * 65)
print("Experiment 4: Scrape arXiv Papers")
print("=" * 65)
print()

papers = scrape_arxiv_abstracts(
    topic="large language models",
    max_results=5,
    save_path=os.path.join(outputs_dir, 'arxiv_papers.json'),
)

Experiment 4: Scrape arXiv Papers

Scraping arXiv: 'large language models' (max 5 papers)

  [1] Unifying Early and Late Dark Energy: Dynamical Requirements and Obstructions...
      Authors: William Giarè, Jeremy Sakstein
      Abstract: We investigate whether early- and late-time dark energy could arise from a single scalar field. Adopting a bottom-up per...

  [2] MobileGym: A Verifiable and Highly Parallel Simulation Platform for Mobile GUI A...
      Authors: Dingbang Wu, Rui Hao, Haiyang Wang...
      Abstract: We present MobileGym, a browser-hosted, lightweight, fully controllable environment for everyday mobile use, targeting i...

  [3] From Model Scaling to System Scaling: Scaling the Harness in Agentic AI...
      Authors: Shangding Gu
      Abstract: This paper studies the next major bottleneck in agentic AI as system scaling, not only model scaling: the design of audi...

  [4] Squeezing Capacity from Multimodal Large Language Models for Subject-driven Gene...
      Author

In [33]:
# TODO 3: Scrape papers on YOUR topic
#
# Choose a topic relevant to your capstone project or interests.
# Scrape 5-10 papers and save the results.

my_topic = "Electrical Engineering"  # e.g., "AI safety", "robotics", "NLP"

print("=" * 65)
print("TODO 3: Custom arXiv Scraping")
print("=" * 65)
print()

my_papers = scrape_arxiv_abstracts(
    topic=my_topic,
    max_results=8,
    save_path=os.path.join(outputs_dir, 'my_arxiv_papers.json'),
)

# Ask Claude to analyze the papers
paper_titles = [p['title'] for p in my_papers]
start = time.time()
response = client.generate(
    prompt=f"""I scraped these {len(my_papers)} papers from arXiv on the topic '{my_topic}':

{chr(10).join(f'{i+1}. {t}' for i, t in enumerate(paper_titles))}

Briefly analyze: What themes do you see? Which 2-3 papers seem most relevant for understanding current trends in this area?""",
    system="You are a research assistant helping analyze academic literature.",
    max_tokens=400,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"\nAnalysis ({elapsed:.1f}s):")
    print(format_response(response, verbose=False))

todo3_reflection = """
[YOUR REFLECTION HERE]

- What topic did you choose and why?
- I chose Electrical Engineering because it is a broad field with many subtopics, and I wanted to see what kind of papers are being published recently in this area. I am particularly interested in how electrical engineering intersects with AI and machine learning, so I wanted to find papers that might touch on those themes.
- Were you surprised by any of the papers found?
- I was surprised to find some papers that applied machine learning techniques to traditional electrical engineering problems, such as signal processing and circuit design. It shows how interdisciplinary the field is becoming, and how AI is influencing even areas that are not traditionally associated with it.
- How could this scraping approach scale to build a real pretraining dataset?
- This scraping approach could be scaled by automating the process to scrape a large number of papers across multiple topics and categories on arXiv. We could set up a pipeline that regularly scrapes new papers, extracts the relevant text (abstracts, introductions), and formats it for LLM pretraining. Additionally, we could use metadata (authors, publication date, categories) to organize the dataset and potentially filter for higher-quality or more relevant papers.
"""

print()
print(todo3_reflection)

TODO 3: Custom arXiv Scraping

Scraping arXiv: 'Electrical Engineering' (max 8 papers)

  [1] TriSplat: Simulation-Ready Feed-Forward 3D Scene Reconstruction...
      Authors: Weijie Wang, Zimu Li, Jinchuan Shi...
      Abstract: Sparse-view 3D reconstruction is increasingly addressed with feed-forward splatting networks that predict explicit primi...

  [2] Prism: A Plug-in Reproducible Infrastructure for Scalable Multimodal Continual I...
      Authors: Jun-Tao Tang, Yu-Cheng Shi, Zhen-Hao Xie...
      Abstract: Multimodal Large Language Models (MLLMs) achieve versatility by reformulating diverse tasks into a unified instruction-f...

  [3] Beyond Summaries: Structure-Aware Labeling of Code Changes with Large Language M...
      Authors: Bar Weiss, Antonio Abu-Nassar, Adi Sosnovich...
      Abstract: Code review is a critical practice in software engineering, yet the growing scale and frequency of code patches in moder...

  [4] Response of a dipolar BEC to Laguerre-Gaussian beam dri

---

## Summary & Reflection

In [34]:
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'
_todo3 = todo3_reflection.strip() if 'todo3_reflection' in dir() else '[TODO 3 not completed yet]'

full_reflection = f"""
### Part 1 - trafilatura Extraction

{_todo1}

---

### Part 2 - Crawl4AI vs trafilatura

{_todo2}

---

### Part 3 - arXiv Paper Dataset

{_todo3}
"""

reflection_file = append_to_reflection(
    notebook="02",
    section_title="Web Scraping & Text Extraction",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ..\outputs\homework_reflection.md

API COST REPORT
Total API calls:     3
Total input tokens:  627
Total output tokens: 1,163
Total cost:          $0.0193

Last 3 calls:
  1. [23:01:50] sonnet -- 182in/400out -- $0.0065
  2. [23:05:43] sonnet -- 182in/400out -- $0.0065
  3. [23:08:42] sonnet -- 263in/363out -- $0.0062


## Notebook 02 Complete!

**What you accomplished:**
- Extracted clean text from web pages with trafilatura
- Compared traditional vs modern (Crawl4AI) extraction approaches
- Built a mini paper dataset from arXiv

**Key concepts:**
- trafilatura removes HTML boilerplate to extract main content
- Crawl4AI produces LLM-ready Markdown with preserved structure
- arXiv API provides structured access to scientific papers

**Next:** Open **Notebook 03: Document OCR & PDF Extraction**